In [34]:
!pip install sqlalchemy pandas

In [35]:
import pandas as pd

from sqlalchemy import (
    create_engine,
    text,
    MetaData,
    Table,
    Column,
    Integer,
    String,
    Float,
    ForeignKey,
    insert,
    update,
    select,
    func
)

from sqlalchemy.orm import (
    declarative_base,
    mapped_column,
    Mapped,
    relationship,
    sessionmaker
)

# Nível 1 — SQL puro com segurança

In [36]:
engine = create_engine("sqlite:///sistema_rh.db")

print("Conexão criada com sucesso!")

Conexão criada com sucesso!


# Passo 2 — Criar a tabela

In [37]:
with engine.begin() as conn:
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS funcionarios (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            nome VARCHAR(100) NOT NULL,
            cargo VARCHAR(100) NOT NULL,
            salario FLOAT NOT NULL
        )
    """))

print("Tabela funcionarios criada!")

Tabela funcionarios criada!


# Passo 3 — Inserção segura

In [38]:
nome_formulario = "Ana Souza"
cargo_formulario = "Desenvolvedor Júnior"
salario_formulario = 3500.00

In [39]:
with engine.begin() as conn:
    conn.execute(
        text("""
            INSERT INTO funcionarios (nome, cargo, salario)
            VALUES (:nome, :cargo, :salario)
        """),
        {
            "nome": nome_formulario,
            "cargo": cargo_formulario,
            "salario": salario_formulario
        }
    )

print("Funcionário inserido com sucesso!")

Funcionário inserido com sucesso!


# Passo 4 — Consultar com Pandas

In [40]:
df_funcionarios = pd.read_sql_query(
    "SELECT * FROM funcionarios",
    engine
)

df_funcionarios

,id,nome,cargo,salario
0,1,Ana Souza,Desenvolvedor Júnior,3500.0
1,2,Ana Souza,Desenvolvedor Júnior,3500.0
2,3,Ana Souza,Desenvolvedor Júnior,3500.0


# Nível 2 — SQLAlchemy Core

In [41]:
metadata = MetaData()

In [42]:
projetos = Table(
    "projetos",
    metadata,

    Column("id", Integer, primary_key=True, autoincrement=True),
    Column("nome", String(100), nullable=False),
    Column("descricao", String(255)),
    Column("responsavel", String(100))
)

In [43]:
metadata.create_all(engine)

print("Tabela projetos criada!")



Tabela projetos criada!


# Passo 2 — Inserção em lote

In [44]:
lista_de_projetos = [
    {
        "nome": "Sistema de Folha",
        "descricao": "Desenvolvimento do sistema de folha de pagamento",
        "responsavel": "Carlos"
    },
    {
        "nome": "Portal RH",
        "descricao": "Portal interno para funcionários",
        "responsavel": "Mariana"
    },
    {
        "nome": "Dashboard Gerencial",
        "descricao": "Dashboard para análise de indicadores",
        "responsavel": "João"
    }
]

In [45]:
with engine.begin() as conn:
    conn.execute(
        insert(projetos),
        lista_de_projetos
    )

print("Projetos inseridos com sucesso!")

Projetos inseridos com sucesso!


In [46]:
df_projetos = pd.read_sql_query(
    "SELECT * FROM projetos",
    engine
)

df_projetos

,id,nome,descricao,responsavel
0,1,Sistema de Folha,Desenvolvimento do sistema de folha de pagamento,Carlos
1,2,Portal RH,Portal interno para funcionários,Mariana
2,3,Dashboard Gerencial,Dashboard para análise de indicadores,João
3,4,Sistema de Folha,Desenvolvimento do sistema de folha de pagamento,Carlos
4,5,Portal RH,Portal interno para funcionários,Mariana
5,6,Dashboard Gerencial,Dashboard para análise de indicadores,João
6,7,Sistema de Folha,Desenvolvimento do sistema de folha de pagamento,Carlos
7,8,Portal RH,Portal interno para funcionários,Mariana
8,9,Dashboard Gerencial,Dashboard para análise de indicadores,João


# Passo 3 — Reajuste salarial

In [47]:
from sqlalchemy import MetaData, Table, update

metadata_funcionarios = MetaData()

funcionarios = Table(
    "funcionarios",
    metadata_funcionarios,
    autoload_with=engine
)

print("Tabela funcionarios carregada!")

Tabela funcionarios carregada!


In [48]:
pd.read_sql_query(
    "SELECT * FROM funcionarios",
    engine
)

,id,nome,cargo,salario
0,1,Ana Souza,Desenvolvedor Júnior,3500.0
1,2,Ana Souza,Desenvolvedor Júnior,3500.0
2,3,Ana Souza,Desenvolvedor Júnior,3500.0


# Passo 4 — Relatório salarial por cargo

In [49]:
consulta_media = (
    select(
        funcionarios.c.cargo,
        func.avg(funcionarios.c.salario).label("media_salarial")
    )
    .group_by(funcionarios.c.cargo)
)

In [50]:
with engine.connect() as conn:
    resultado = conn.execute(consulta_media)

    for linha in resultado:
        print(linha)

('Desenvolvedor Júnior', 3500.0)


# Nível 3 — ORM

In [51]:
Base = declarative_base()

In [52]:
class Departamento(Base):
    __tablename__ = "departamentos"

    id: Mapped[int] = mapped_column(
        Integer,
        primary_key=True,
        autoincrement=True
    )

    nome: Mapped[str] = mapped_column(
        String(100),
        nullable=False
    )

    funcionarios: Mapped[list["FuncionarioORM"]] = relationship(
        back_populates="departamento",
        cascade="all, delete-orphan"
    )

In [53]:
class FuncionarioORM(Base):
    __tablename__ = "funcionarios_orm"

    id: Mapped[int] = mapped_column(
        Integer,
        primary_key=True,
        autoincrement=True
    )

    nome: Mapped[str] = mapped_column(
        String(100),
        nullable=False
    )

    cargo: Mapped[str] = mapped_column(
        String(100),
        nullable=False
    )

    salario: Mapped[float] = mapped_column(
        Float,
        nullable=False
    )

    departamento_id: Mapped[int] = mapped_column(
        ForeignKey("departamentos.id")
    )

    departamento: Mapped["Departamento"] = relationship(
        back_populates="funcionarios"
    )

Passo 2 — Criar as tabelas

In [54]:
Base.metadata.create_all(engine)

print("Tabelas ORM criadas!")

Tabelas ORM criadas!


Passo 3 — Criar a sessão e inserir dados

In [55]:
Session = sessionmaker(bind=engine)

In [56]:
sessao = Session()

In [57]:
departamento_ti = Departamento(
    nome="TI"
)

In [58]:
funcionario1 = FuncionarioORM(
    nome="Pedro",
    cargo="Desenvolvedor",
    salario=5000.00
)

funcionario2 = FuncionarioORM(
    nome="Juliana",
    cargo="Analista de Sistemas",
    salario=4500.00
)

In [59]:
departamento_ti.funcionarios.append(funcionario1)
departamento_ti.funcionarios.append(funcionario2)

In [60]:
sessao.add(departamento_ti)

In [61]:
sessao.commit()

print("Departamento e funcionários salvos!")

Departamento e funcionários salvos!


Passo 4 — Consultar funcionários do departamento de TI

In [62]:
consulta = (
    select(FuncionarioORM)
    .join(FuncionarioORM.departamento)
    .where(Departamento.nome == "TI")
)

In [63]:
funcionarios_ti = sessao.execute(
    consulta
).scalars().all()

In [64]:
for funcionario in funcionarios_ti:
    print(
        funcionario.nome,
        "-",
        funcionario.cargo,
        "- R$",
        funcionario.salario
    )

Pedro - Desenvolvedor - R$ 5000.0
Juliana - Analista de Sistemas - R$ 4500.0


In [65]:
sessao.close()